In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

#1. User Interaction
#Prompt the user to enter a job title
search = input("Enter the job title to search (e.g., Data Scientist): ")

#2. Web Scraping
url = "https://www.careerjunction.co.za/jobs/results"
search_url = "{}?keywords={}".format(url, search.replace(" ", "+"))

#Send a request to the website
r = requests.get(url)
soup = BeautifulSoup(r.text, 'html.parser')

# Function to extract data from a single page of search results
def extract_job_data(url):
    try:
        r = requests.get(url)
        r.raise_for_status()  # Raise an exception for bad status codes

        soup = BeautifulSoup(r.content, "html.parser")

        # Extract data 
        job_listings = soup.find_all("div", class_ = "job-listing")

        data = []
        for listing in job_listings:
            title = listing.find("h2", class_ = "job-title").text.strip()
            recruiter = listing.find("p", class_ = "company-name").text.strip()
            salary = listing.find("p", class_ = "salary").text.strip()
            position = listing.find("p", class_ = "position-type").text.strip()
            location = listing.find("p", class_ = "location").text.strip()
            date_posted = listing.find("p", class_ = "date-posted").text.strip()

            #create the data frame
            data.append([title, recruiter, salary, position, location, date_posted])

        return pd.DataFrame(data, columns = ["Title", "Recruiter", "Salary", "Position", "Location", "Date Posted"])

    except requests.exceptions.RequestException as e:
        print("Error during request: " + str(e))
        return None
    except Exception as e:
        print("Error during scraping: " + str(e))
        return None

# Get the first page of results
job_data_df = extract_job_data(search_url)

#Save the datframe into a csv file
if job_data_df is not None:
    #Data Storage and Saving
    csv_filename = search.replace(" ", "_") + "-job-results.csv"
    job_data_df.to_csv(csv_filename, index = False)
    print("Data saved to " + csv_filename)
else:
    print("No data extracted or error occurred.")

Enter the job title to search (e.g., Data Scientist):  Software Engineer


Data saved to Software_Engineer-job-results.csv
